# Contact: pressing a sphere into a soft slab

**The question.** Real soft-body work is full of contact. When a rigid sphere presses into a soft slab, what does each solver actually give you?

**The test.** Push a rigid sphere into a clamped slab in small steps. FEM resolves the contact with a penalty / Augmented-Lagrangian law written in pure UFL (no C++); Newton's XPBD enforces non-penetration by positional projection.

**The verdict.** FEM produces a **calibrated contact-force curve** (anchored against the analytic Hertz solution); **XPBD exposes no comparable force** — it only gives deformation. So the honest comparison axis is the **deformed shape**, and FEM's force curve is something the fast solver simply cannot provide. The FEM variants form a gradation from coarse to accurate contact: `tet kn×5 penalty` → `tet kn×50 penalty` → `tet kn×10 AL` → `hex kn×10 AL` (locking-free hex + Augmented Lagrangian, penetration ≈ 0).

In [ ]:
%matplotlib inline
import os

import matplotlib.pyplot as plt
import numpy as np

from common import params

fem = np.load(params.FEM_INDENT_NPZ)
newton = np.load(params.NEWTON_INDENT_NPZ) if os.path.exists(params.NEWTON_INDENT_NPZ) else None
slugs = [k[3:] for k in fem.files if k.startswith("uz_")]
d = fem["deltas"] * 1000.0
print("FEM variants:", slugs, "| Newton present:", newton is not None)

## The scene — what we're actually simulating

A clamped soft slab (1.0 × 1.0 × 0.4 m) with a rigid sphere (R = 0.30 m) lowered onto its top face in small steps. Below is Newton's slab at maximum indentation — rendered from the **real mesh** once `run_indentation` has run (a setup schematic otherwise) — with the sphere as a wireframe. The sphere is treated **analytically** (its gap to the deformed surface is closed-form), so there is no mesh-to-mesh contact search; that is what keeps the whole contact law inside UFL.

In [ ]:
from compare import scene

fig = plt.figure(figsize=(5, 5))
ax = fig.add_subplot(111, projection="3d")
if newton is not None and "tet_indices" in newton.files:
    norm, lab = scene.render(ax, newton["rest_q"], newton["final_q"], newton["tet_indices"],
                             ghost_rest=False, title="Newton XPBD - indented slab (real mesh)")
    scene.add_sphere(ax, newton["sphere_c"], float(newton["sphere_r"]))
    scene.add_colorbar(fig, ax, norm, lab)
else:
    nx, ny, nz = params.INDENT_DIM
    h = params.INDENT_CELL
    R = params.INDENT_SPHERE_R
    Lx, Ly, Lz = nx * h, ny * h, nz * h
    scene.draw_box(ax, (0, 0, 0), (Lx, Ly, Lz), frame_pts=[[0, 0, 0], [Lx, Ly, Lz + R + 0.05]])
    scene.add_sphere(ax, (Lx / 2.0, Ly / 2.0, Lz + R - params.INDENT_MAX), R)
    ax.set_title("Setup schematic (run newton_run.run_indentation for the real dimple)")
ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]"); ax.set_zlabel("z [m]")
ax.view_init(elev=18, azim=-60)
plt.tight_layout(); plt.show()

## 1. The verdict — a contact force curve only FEM can give

As the sphere presses in, FEM assembles the **total contact force** at each depth; the dashed line is the analytic **Hertz** solution for a rigid sphere on an elastic half-space (an approximate anchor — our slab is finite and soft). XPBD enforces non-penetration positionally and produces **no comparable force**, so there is no XPBD curve here — and that absence *is* the headline.

In [ ]:
plt.figure(figsize=(6, 5))
for s in slugs:
    plt.plot(d, fem["f_" + s], "o-", label="FEM " + s)
plt.plot(d, fem["f_hertz"], "k--", lw=1.5, label="Hertz (analytic anchor)")
plt.xlabel("indentation depth [mm]"); plt.ylabel("contact force [N]")
plt.legend(); plt.grid(alpha=0.3)
plt.title("Indentation - contact force (XPBD exposes none)"); plt.show()

## 2. The honest comparison — the deformed dimple

Since only deformation is common to both solvers, this is where they meet: the vertical displacement of the top surface along a line through the contact centre. FEM variants and (if run) Newton XPBD are overlaid — the same physical dimple, seen by a fast positional solver and by implicit FEM.

In [ ]:
plt.figure(figsize=(6, 5))
fx = (fem["line_x"] - float(fem["cx"])) * 1000.0
for s in slugs:
    plt.plot(fx, fem["uz_" + s] * 1000.0, label="FEM " + s)
if newton is not None:
    nx_ = (newton["line_x"] - float(newton["cx"])) * 1000.0
    plt.plot(nx_, newton["uz_line"] * 1000.0, "k-o", ms=3, label="Newton XPBD")
plt.xlabel("x - centre [mm]"); plt.ylabel("vertical displacement u_z [mm]")
plt.legend(); plt.grid(alpha=0.3)
plt.title("Indentation - deformed dimple (FEM vs Newton)"); plt.show()

## 3. The accuracy gradation — residual penetration (penalty vs Augmented Lagrangian)

The clearest "coarse → accurate" axis. Plain **penalty** leaves a penetration that depends on the stiffness `kn`; the **Augmented-Lagrangian** Uzawa loop drives it toward zero at the *same* modest `kn` — approaching the exact non-penetration constraint. Watch `tet kn×5 penalty` (most penetration) collapse to the `AL` variants (≈ 0).

In [ ]:
plt.figure(figsize=(6, 5))
for s in slugs:
    if ("pen_" + s) in fem.files:
        plt.plot(d, fem["pen_" + s] * 1000.0, "o-", label="FEM " + s)
plt.xlabel("indentation depth [mm]"); plt.ylabel("max residual penetration [mm]")
plt.legend(); plt.grid(alpha=0.3)
plt.title("Indentation - residual penetration (AL -> ~0)"); plt.show()

## 4. Internal (strain) energy vs indentation

How much elastic energy the slab stores as the sphere presses in — comparable across variants and Newton because it uses the same Neo-Hookean density on the same kind of mesh.

In [ ]:
plt.figure(figsize=(6, 5))
for s in slugs:
    plt.plot(d, fem["e_strain_" + s], "o-", label="FEM " + s)
if newton is not None:
    plt.plot(d, newton["e_strain"], "k-o", ms=3, label="Newton XPBD")
plt.xlabel("indentation depth [mm]"); plt.ylabel("strain energy [J]")
plt.legend(); plt.grid(alpha=0.3)
plt.title("Indentation - internal energy"); plt.show()

## 5. Contact (penalty) energy — FEM

The penalty potential `1/2 kn <-g>+^2` stored in the contact layer. A stiffer `kn` stores less of it (less penetration); the Augmented-Lagrangian variant keeps it small even at modest `kn`.

In [ ]:
plt.figure(figsize=(6, 5))
for s in slugs:
    plt.plot(d, fem["e_contact_" + s], "o-", label="FEM " + s)
plt.xlabel("indentation depth [mm]"); plt.ylabel("contact (penalty) energy [J]")
plt.legend(); plt.grid(alpha=0.3)
plt.title("Indentation - contact energy (FEM penalty)"); plt.show()

## 6. Newton XPBD — penetration & settling diagnostic

XPBD's own quality check: how much the body penetrates the sphere, and whether the positional solve settled (residual kinetic energy → 0) at each indentation step.

In [ ]:
if newton is not None:
    fig, ax1 = plt.subplots(figsize=(6, 4))
    ax1.plot(d, newton["penetration"] * 1000.0, "tab:red", marker="o", ms=3)
    ax1.set_xlabel("indentation depth [mm]"); ax1.set_ylabel("max penetration [mm]", color="tab:red")
    ax2 = ax1.twinx(); ax2.semilogy(d, np.maximum(newton["ke"], 1e-12), "tab:green", alpha=0.7)
    ax2.set_ylabel("residual KE [J] (settled ~ 0)", color="tab:green")
    plt.title("Indentation - Newton penetration & settle KE"); plt.tight_layout(); plt.show()
else:
    print("No Newton indentation result yet -- run: python -m newton_run.run_indentation")

## 7. Computation time (wall-clock)

The cost of each variant (and Newton), per the same trade-off as the hanging bar: the heavier the contact model (more penalty iterations, Augmented-Lagrangian outer loops), the more time it takes.

In [ ]:
ft = {s: float(fem["time_" + s]) for s in slugs if ("time_" + s) in fem.files}
if newton is not None and "wall_time" in newton.files:
    ft["Newton XPBD"] = float(newton["wall_time"])
for k, v in ft.items():
    print(f"{k:16s}: {v:8.3f} s")
plt.figure(figsize=(6, 4))
plt.bar(list(ft), list(ft.values()))
plt.ylabel("wall time [s]"); plt.title("Indentation - computation time"); plt.xticks(rotation=20)
plt.tight_layout(); plt.show()

## Takeaway

* FEM gives a **calibrated contact-force curve** that follows the Hertz `δ^{3/2}` trend and deviates as the contact radius grows relative to the finite slab — a curve **XPBD cannot produce** (§1).
* Across the gradation, **stiffer penalty → less penetration**, and the **Augmented Lagrangian drives penetration ≈ 0** at modest `kn` (§3); on locking-free hex it is the most accurate of the four.
* So the solvers are honestly compared on the **deformed dimple** (§2). The missing XPBD force curve *is* the difference between a fast positional projection and an implicit FEM contact solve.
* Friction is deliberately left out here (frictionless keeps the Hertz comparison clean); it is its own scenario in `40_friction.ipynb`.